#  Structuring Machine Learning Projects (End-to-End)

## Learning Objectives
- Understand the complete ML project lifecycle
- Master data preprocessing and feature engineering
- Learn proper model evaluation metrics and techniques
- Explore deployment considerations and MLOps

## 3.1 ML Project Lifecycle

### Key Stages:
1. **Problem Definition** - Clear objectives and success metrics
2. **Data Collection** - Gather relevant data
3. **Data Exploration** - Understand data characteristics
4. **Data Preprocessing** - Clean and prepare data
5. **Model Development** - Build and train models
6. **Model Evaluation** - Assess performance properly
7. **Deployment** - Put model into production
8. **Monitoring** - Track performance over time

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, classification_report
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)

print("Libraries imported successfully!")

## 3.2 Data Collection and Exploration

In [ ]:
# Load a real dataset for our end-to-end example
# California Housing dataset for regression
housing = fetch_california_housing()
X_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
y_housing = housing.target

print("California Housing Dataset:")
print(f"Shape: {X_housing.shape}")
print(f"\nFeature names: {list(housing.feature_names)}")
print(f"\nTarget description: {housing.DESCR[:200]}...")

# Basic statistics
print("\nBasic Statistics:")
print(X_housing.describe())

# Check for missing values
print(f"\nMissing values per feature:")
print(X_housing.isnull().sum())

In [ ]:
# Data visualization and exploration
plt.figure(figsize=(20, 15))

# Distribution of target variable
plt.subplot(3, 3, 1)
plt.hist(y_housing, bins=50, alpha=0.7)
plt.title('Distribution of Housing Prices')
plt.xlabel('Median House Value ($100,000s)')
plt.ylabel('Frequency')

# Correlation heatmap
plt.subplot(3, 3, 2)
correlation_matrix = X_housing.corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')

# Scatter plots for key features
features_to_plot = ['MedInc', 'HouseAge', 'AveRooms', 'Population']
for i, feature in enumerate(features_to_plot):
    plt.subplot(3, 3, i + 3)
    plt.scatter(X_housing[feature], y_housing, alpha=0.5)
    plt.title(f'{feature} vs Price')
    plt.xlabel(feature)
    plt.ylabel('Price')

# Box plots for outlier detection
for i, feature in enumerate(features_to_plot):
    plt.subplot(3, 3, i + 7)
    plt.boxplot(X_housing[feature])
    plt.title(f'{feature} Boxplot')
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

# Feature correlations with target
correlations = X_housing.corrwith(pd.Series(y_housing)).sort_values(ascending=False)
print("\nFeature correlations with target:")
print(correlations)

## 3.3 Data Preprocessing Pipeline

In [ ]:
class DataPreprocessor:
    def __init__(self):
        self.scalers = {}
        self.feature_selector = None
        self.selected_features = None
    
    def detect_outliers(self, X, method='iqr', factor=1.5):
        """Detect outliers using IQR method"""
        if isinstance(X, pd.DataFrame):
            X = X.values
        
        Q1 = np.percentile(X, 25, axis=0)
        Q3 = np.percentile(X, 75, axis=0)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - factor * IQR
        upper_bound = Q3 + factor * IQR
        
        outliers = (X < lower_bound) | (X > upper_bound)
        return outliers.any(axis=1)
    
    def handle_outliers(self, X, y, method='remove'):
        """Handle outliers in the dataset"""
        outlier_mask = self.detect_outliers(X)
        
        if method == 'remove':
            return X[~outlier_mask], y[~outlier_mask]
        elif method == 'cap':
            X_capped = X.copy()
            if isinstance(X_capped, pd.DataFrame):
                for col in X_capped.columns:
                    Q1 = X_capped[col].quantile(0.25)
                    Q3 = X_capped[col].quantile(0.75)
                    IQR = Q3 - Q1
                    lower_bound = Q1 - 1.5 * IQR
                    upper_bound = Q3 + 1.5 * IQR
                    X_capped[col] = X_capped[col].clip(lower_bound, upper_bound)
            return X_capped, y
        else:
            return X, y
    
    def scale_features(self, X, method='standard', fit=True):
        """Scale features using different methods"""
        if method == 'standard':
            scaler = StandardScaler()
        elif method == 'minmax':
            scaler = MinMaxScaler()
        elif method == 'robust':
            scaler = RobustScaler()
        else:
            raise ValueError(f"Unknown scaling method: {method}")
        
        if fit:
            X_scaled = scaler.fit_transform(X)
            self.scalers[method] = scaler
        else:
            X_scaled = self.scalers[method].transform(X)
        
        return X_scaled
    
    def select_features(self, X, y, method='univariate', k=10):
        """Feature selection"""
        if method == 'univariate':
            selector = SelectKBest(score_func=f_regression, k=k)
            X_selected = selector.fit_transform(X, y)
            self.feature_selector = selector
            
            # Get selected feature names if X is DataFrame
            if isinstance(X, pd.DataFrame):
                selected_mask = selector.get_support()
                self.selected_features = X.columns[selected_mask].tolist()
            
            return X_selected
        else:
            raise ValueError(f"Unknown feature selection method: {method}")
    
    def fit_transform(self, X, y, scale_method='standard', 
                    handle_outliers_method='remove', 
                    feature_selection_method='univariate', k=10):
        """Complete preprocessing pipeline"""
        # Handle outliers
        X_clean, y_clean = self.handle_outliers(X, y, method=handle_outliers_method)
        
        # Feature selection
        if feature_selection_method:
            X_selected = self.select_features(X_clean, y_clean, 
                                         method=feature_selection_method, k=k)
        else:
            X_selected = X_clean
        
        # Scale features
        X_processed = self.scale_features(X_selected, method=scale_method, fit=True)
        
        return X_processed, y_clean
    
    def transform(self, X):
        """Transform new data using fitted preprocessing"""
        # Apply feature selection if fitted
        if self.feature_selector is not None:
            X = self.feature_selector.transform(X)
        
        # Apply scaling if fitted
        if self.scalers:
            # Use the first available scaler
            scale_method = list(self.scalers.keys())[0]
            X = self.scale_features(X, method=scale_method, fit=False)
        
        return X

# Test the preprocessing pipeline
preprocessor = DataPreprocessor()

print("Original data shape:", X_housing.shape)
print("Original target shape:", y_housing.shape)

# Apply preprocessing
X_processed, y_processed = preprocessor.fit_transform(
    X_housing, y_housing, 
    scale_method='standard',
    handle_outliers_method='remove',
    feature_selection_method='univariate',
    k=8
)

print(f"\nProcessed data shape: {X_processed.shape}")
print(f"Processed target shape: {y_processed.shape}")
print(f"Selected features: {preprocessor.selected_features}")

## 3.4 Model Development and Evaluation

In [ ]:
# Split data properly
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
}

# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'model': model,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_test_pred': y_test_pred
    }
    
    print(f"Train MSE: {train_mse:.4f}")
    print(f"Test MSE: {test_mse:.4f}")
    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# Visualize model performance
plt.figure(figsize=(15, 10))

# Model comparison - R² scores
plt.subplot(2, 3, 1)
model_names = list(results.keys())
train_r2_scores = [results[name]['train_r2'] for name in model_names]
test_r2_scores = [results[name]['test_r2'] for name in model_names]

x = np.arange(len(model_names))
width = 0.35

plt.bar(x - width/2, train_r2_scores, width, label='Train R²', alpha=0.8)
plt.bar(x + width/2, test_r2_scores, width, label='Test R²', alpha=0.8)
plt.xlabel('Models')
plt.ylabel('R² Score')
plt.title('Model Performance Comparison')
plt.xticks(x, model_names)
plt.legend()
plt.grid(True, alpha=0.3)

# MSE comparison
plt.subplot(2, 3, 2)
train_mse_scores = [results[name]['train_mse'] for name in model_names]
test_mse_scores = [results[name]['test_mse'] for name in model_names]

plt.bar(x - width/2, train_mse_scores, width, label='Train MSE', alpha=0.8)
plt.bar(x + width/2, test_mse_scores, width, label='Test MSE', alpha=0.8)
plt.xlabel('Models')
plt.ylabel('MSE')
plt.title('Model MSE Comparison')
plt.xticks(x, model_names)
plt.legend()
plt.grid(True, alpha=0.3)

# Cross-validation scores
plt.subplot(2, 3, 3)
cv_means = [results[name]['cv_mean'] for name in model_names]
cv_stds = [results[name]['cv_std'] for name in model_names]

plt.bar(model_names, cv_means, yerr=cv_stds, capsize=5, alpha=0.8)
plt.xlabel('Models')
plt.ylabel('CV R² Score')
plt.title('Cross-Validation Performance')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Prediction vs Actual plots
for i, name in enumerate(model_names):
    plt.subplot(2, 3, i + 4)
    y_test_pred = results[name]['y_test_pred']
    
    plt.scatter(y_test, y_test_pred, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title(f'{name} - Predictions vs Actual')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3.5 Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestRegressor(random_state=42)

print("Performing Grid Search for Random Forest...")
grid_search = GridSearchCV(
    rf, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Evaluate best model
best_rf = grid_search.best_estimator_
y_test_pred_best = best_rf.predict(X_test)
test_r2_best = r2_score(y_test, y_test_pred_best)
test_mse_best = mean_squared_error(y_test, y_test_pred_best)

print(f"\nBest model test R²: {test_r2_best:.4f}")
print(f"Best model test MSE: {test_mse_best:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': preprocessor.selected_features,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.gca().invert_yaxis()
plt.show()

## 3.6 Model Evaluation Metrics

In [ ]:
# Let's also demonstrate classification metrics
# Load breast cancer dataset
cancer = load_breast_cancer()
X_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_cancer = cancer.target

print("Breast Cancer Dataset:")
print(f"Shape: {X_cancer.shape}")
print(f"Classes: {cancer.target_names}")
print(f"Class distribution: {np.bincount(y_cancer)}")

# Preprocess classification data
preprocessor_clf = DataPreprocessor()
X_cancer_processed, y_cancer_processed = preprocessor_clf.fit_transform(
    X_cancer, y_cancer,
    scale_method='standard',
    handle_outliers_method='remove',
    feature_selection_method='univariate',
    k=15
)

# Split data
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer_processed, y_cancer_processed, test_size=0.2, random_state=42, stratify=y_cancer_processed
)

# Train classification models
clf_models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

clf_results = {}

for name, model in clf_models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_c, y_train_c)
    
    # Predictions
    y_train_pred = model.predict(X_train_c)
    y_test_pred = model.predict(X_test_c)
    y_test_proba = model.predict_proba(X_test_c)[:, 1]
    
    # Metrics
    train_acc = model.score(X_train_c, y_train_c)
    test_acc = model.score(X_test_c, y_test_c)
    auc_score = roc_auc_score(y_test_c, y_test_proba)
    
    clf_results[name] = {
        'model': model,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'auc': auc_score,
        'y_test_pred': y_test_pred,
        'y_test_proba': y_test_proba
    }
    
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test_c, y_test_pred, target_names=cancer.target_names))

In [ ]:
# Visualize classification results
plt.figure(figsize=(15, 10))

# Accuracy comparison
plt.subplot(2, 3, 1)
clf_names = list(clf_results.keys())
train_accs = [clf_results[name]['train_acc'] for name in clf_names]
test_accs = [clf_results[name]['test_acc'] for name in clf_names]

x = np.arange(len(clf_names))
width = 0.35

plt.bar(x - width/2, train_accs, width, label='Train Accuracy', alpha=0.8)
plt.bar(x + width/2, test_accs, width, label='Test Accuracy', alpha=0.8)
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Classification Accuracy Comparison')
plt.xticks(x, clf_names)
plt.legend()
plt.grid(True, alpha=0.3)

# AUC scores
plt.subplot(2, 3, 2)
auc_scores = [clf_results[name]['auc'] for name in clf_names]
plt.bar(clf_names, auc_scores, alpha=0.8)
plt.xlabel('Models')
plt.ylabel('AUC Score')
plt.title('AUC Score Comparison')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Confusion matrices
for i, name in enumerate(clf_names):
    plt.subplot(2, 3, i + 3)
    y_test_pred = clf_results[name]['y_test_pred']
    cm = confusion_matrix(y_test_c, y_test_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=cancer.target_names, yticklabels=cancer.target_names)
    plt.title(f'{name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

# ROC curves
plt.subplot(2, 3, 6)
for name in clf_names:
    y_test_proba = clf_results[name]['y_test_proba']
    fpr, tpr, _ = roc_curve(y_test_c, y_test_proba)
    auc_score = clf_results[name]['auc']
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3.7 Deployment Considerations

In [ ]:
import joblib
import json
from datetime import datetime

class MLModelDeployer:
    def __init__(self, model, preprocessor, model_name="model"):
        self.model = model
        self.preprocessor = preprocessor
        self.model_name = model_name
        self.metadata = {
            'model_type': type(model).__name__,
            'created_at': datetime.now().isoformat(),
            'features': preprocessor.selected_features if hasattr(preprocessor, 'selected_features') else None,
            'target_type': 'regression' if hasattr(model, 'predict') and not hasattr(model, 'predict_proba') else 'classification'
        }
    
    def save_model(self, filepath):
        """Save model and preprocessing pipeline"""
        model_data = {
            'model': self.model,
            'preprocessor': self.preprocessor,
            'metadata': self.metadata
        }
        
        joblib.dump(model_data, filepath)
        print(f"Model saved to {filepath}")
    
    def load_model(self, filepath):
        """Load model and preprocessing pipeline"""
        model_data = joblib.load(filepath)
        self.model = model_data['model']
        self.preprocessor = model_data['preprocessor']
        self.metadata = model_data['metadata']
        print(f"Model loaded from {filepath}")
        return self.model, self.preprocessor
    
    def predict(self, X):
        """Make predictions on new data"""
        # Preprocess input
        X_processed = self.preprocessor.transform(X)
        
        # Make prediction
        if hasattr(self.model, 'predict_proba'):
            predictions = self.model.predict_proba(X_processed)
        else:
            predictions = self.model.predict(X_processed)
        
        return predictions
    
    def get_model_info(self):
        """Get model information"""
        info = {
            'model_name': self.model_name,
            'model_type': self.metadata['model_type'],
            'created_at': self.metadata['created_at'],
            'target_type': self.metadata['target_type'],
            'num_features': len(self.metadata['features']) if self.metadata['features'] else 'Unknown'
        }
        return info

# Deploy the best regression model
best_rf_regressor = best_rf
deployer = MLModelDeployer(best_rf_regressor, preprocessor, "california_housing_rf")

# Save the model
deployer.save_model('california_housing_model.joblib')

# Get model info
model_info = deployer.get_model_info()
print("\nModel Information:")
for key, value in model_info.items():
    print(f"{key}: {value}")

# Test prediction with new data
print("\nTesting prediction with sample data...")
sample_data = X_housing.head(5)  # Use first 5 samples as "new" data
predictions = deployer.predict(sample_data)
print(f"Sample predictions: {predictions}")
print(f"Actual values: {y_housing[:5]}")

## 3.8 Monitoring and Maintenance

In [ ]:
class ModelMonitor:
    def __init__(self, model, preprocessor, threshold=0.1):
        self.model = model
        self.preprocessor = preprocessor
        self.threshold = threshold
        self.performance_history = []
        self.drift_history = []
    
    def calculate_data_drift(self, reference_data, current_data):
        """Calculate data drift between reference and current data"""
        # Simple drift detection using statistical measures
        drift_scores = {}
        
        for col in reference_data.columns:
            ref_mean = reference_data[col].mean()
            curr_mean = current_data[col].mean()
            ref_std = reference_data[col].std()
            curr_std = current_data[col].std()
            
            # Calculate drift score (simple approach)
            mean_drift = abs(ref_mean - curr_mean) / (ref_std + 1e-8)
            std_drift = abs(ref_std - curr_std) / (ref_std + 1e-8)
            
            drift_scores[col] = {
                'mean_drift': mean_drift,
                'std_drift': std_drift,
                'overall_drift': (mean_drift + std_drift) / 2
            }
        
        return drift_scores
    
    def monitor_performance(self, X_new, y_true, timestamp=None):
        """Monitor model performance on new data"""
        if timestamp is None:
            timestamp = datetime.now()
        
        # Make predictions
        y_pred = self.model.predict(self.preprocessor.transform(X_new))
        
        # Calculate metrics
        mse = mean_squared_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        
        performance_record = {
            'timestamp': timestamp,
            'mse': mse,
            'r2': r2,
            'sample_size': len(y_true)
        }
        
        self.performance_history.append(performance_record)
        
        return performance_record
    
    def check_performance_degradation(self):
        """Check if performance has degraded"""
        if len(self.performance_history) < 2:
            return False, "Insufficient data"
        
        latest = self.performance_history[-1]
        baseline = self.performance_history[0]
        
        # Check R² degradation
        r2_degradation = (baseline['r2'] - latest['r2']) / baseline['r2']
        
        if r2_degradation > self.threshold:
            return True, f"R² degraded by {r2_degradation:.2%}"
        
        return False, "Performance stable"
    
    def generate_report(self):
        """Generate monitoring report"""
        if not self.performance_history:
            return "No monitoring data available"
        
        latest = self.performance_history[-1]
        degradation_detected, message = self.check_performance_degradation()
        
        report = f"""
Model Monitoring Report
=====================
Latest Performance (Date: {latest['timestamp']}):
- MSE: {latest['mse']:.4f}
- R²: {latest['r2']:.4f}
- Sample Size: {latest['sample_size']}

Performance Degradation: {'DETECTED' if degradation_detected else 'NONE'}
Message: {message}

Total Monitoring Records: {len(self.performance_history)}
"""
        return report

# Simulate monitoring
monitor = ModelMonitor(best_rf_regressor, preprocessor, threshold=0.05)

# Simulate performance over time
print("Simulating model monitoring...")

# Initial performance (baseline)
baseline_perf = monitor.monitor_performance(X_test, y_test)
print(f"Baseline R²: {baseline_perf['r2']:.4f}")

# Simulate degraded performance (add noise)
X_degraded = X_test + np.random.normal(0, 0.5, X_test.shape)
y_degraded = y_test + np.random.normal(0, 0.3, y_test.shape)

degraded_perf = monitor.monitor_performance(X_degraded, y_degraded)
print(f"Degraded R²: {degraded_perf['r2']:.4f}")

# Check for degradation
degradation_detected, message = monitor.check_performance_degradation()
print(f"\nDegradation Detected: {degradation_detected}")
print(f"Message: {message}")

# Generate full report
print("\n" + monitor.generate_report())

## 3.9 Key Takeaways

### Project Structure
- **Problem Definition**: Clear objectives and success metrics are crucial
- **Data Quality**: Garbage in, garbage out - invest in data preprocessing
- **Iterative Process**: ML projects are iterative, not linear

### Data Preprocessing
- **Outlier Detection**: Remove or handle outliers appropriately
- **Feature Scaling**: Essential for many algorithms
- **Feature Selection**: Reduces overfitting and improves interpretability

### Model Evaluation
- **Proper Validation**: Use cross-validation, not just train-test split
- **Multiple Metrics**: Different metrics tell different stories
- **Baseline Models**: Always compare against simple baselines

### Deployment Considerations
- **Model Versioning**: Track model versions and performance
- **Monitoring**: Continuously monitor model performance in production
- **Drift Detection**: Watch for data and concept drift

## Exercises

1. **Complete Pipeline**: Build an end-to-end ML pipeline for a different dataset
2. **Advanced Preprocessing**: Implement more sophisticated feature engineering
3. **Model Selection**: Compare more algorithms and use automated selection
4. **Deployment**: Create a simple API for model serving
5. **Monitoring Dashboard**: Build a dashboard to monitor model performance over time